# RandomForest Fraud Detection Experiment
IEEE-CIS Fraud Detection Competition

## 0. Setup & MLflow Configuration

In [ ]:
import os, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from kaggle_secrets import UserSecretsClient


DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
DAGSHUB_USERNAME = 'delibaaa'
REPO_NAME        = 'DELIBA-ML-ASSIGNMENT-2'
os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN

mlflow.set_tracking_uri(f'https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow')
mlflow.set_experiment('RandomForest_Training')
print('Setup complete')

## 1. Data Loading

In [ ]:
DATA_PATH = '/kaggle/input/ieee-fraud-detection'
train = pd.read_csv(f'{DATA_PATH}/train_transaction.csv').merge(
        pd.read_csv(f'{DATA_PATH}/train_identity.csv'), on='TransactionID', how='left')
test  = pd.read_csv(f'{DATA_PATH}/test_transaction.csv').merge(
        pd.read_csv(f'{DATA_PATH}/test_identity.csv'), on='TransactionID', how='left')
print(f'Train: {train.shape} | Test: {test.shape}')

## 2. Cleaning

In [ ]:
with mlflow.start_run(run_name='RandomForest_Cleaning'):
    nan_pct = train.isnull().mean()
    high_nan_cols = nan_pct[nan_pct > 0.9].index.tolist()
    train_clean = train.drop(columns=high_nan_cols)
    test_clean  = test.drop(columns=high_nan_cols)

    constant_cols = [c for c in train_clean.columns if train_clean[c].nunique(dropna=False) <= 1]
    train_clean = train_clean.drop(columns=constant_cols)
    test_clean  = test_clean.drop(columns=constant_cols, errors='ignore')

    mlflow.log_param('nan_strategy', 'median_imputer_in_pipeline')
    mlflow.log_param('dropped_high_nan_cols', len(high_nan_cols))
    mlflow.log_param('dropped_constant_cols', len(constant_cols))
    mlflow.log_metric('cols_after_cleaning', train_clean.shape[1])
    print(f'After cleaning: {train_clean.shape}')

## 3. Feature Engineering

In [ ]:
with mlflow.start_run(run_name='RandomForest_Feature_Engineering'):

    def feature_engineering(df):
        df = df.copy()
        df['TransactionHour']    = (df['TransactionDT'] // 3600 % 24).astype(int)
        df['TransactionDay']     = (df['TransactionDT'] // (3600*24)).astype(int)
        df['TransactionWeekDay'] = (df['TransactionDay'] % 7).astype(int)
        df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
        for col in ['card1', 'card2', 'addr1']:
            if col in df.columns:
                df[f'{col}_freq'] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)
        if 'card1' in df.columns:
            df['card1_amt_mean'] = df.groupby('card1')['TransactionAmt'].transform('mean')
        return df

    train_fe = feature_engineering(train_clean)
    test_fe  = feature_engineering(test_clean)

    mlflow.log_metric('features_after_fe', train_fe.shape[1])
    print(f'After FE: {train_fe.shape}')

## 4. Feature Selection

In [ ]:
with mlflow.start_run(run_name='RandomForest_Feature_Selection'):

    TARGET = 'isFraud'
    DROP_COLS = ['TransactionID', 'TransactionDT', TARGET]
    feature_cols = [c for c in train_fe.columns if c not in DROP_COLS]

    cat_cols = train_fe[feature_cols].select_dtypes(include='object').columns.tolist()
    num_cols = train_fe[feature_cols].select_dtypes(exclude='object').columns.tolist()

    X_temp = train_fe[feature_cols].copy()
    for col in cat_cols:
        X_temp[col] = X_temp[col].astype('category').cat.codes
    X_temp[num_cols] = X_temp[num_cols].fillna(X_temp[num_cols].median())
    X_temp[cat_cols] = X_temp[cat_cols].fillna(-1)

    quick_rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    quick_rf.fit(X_temp, train_fe[TARGET])

    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': quick_rf.feature_importances_
    }).sort_values('importance', ascending=False)

    selected_features = importance_df.head(100)['feature'].tolist()

    importance_df.to_csv('/tmp/rf_feature_importance.csv', index=False)
    mlflow.log_artifact('/tmp/rf_feature_importance.csv')
    mlflow.log_param('selected_features_count', len(selected_features))
    mlflow.log_param('selection_method', 'rf_importance_top100')
    print(f'Selected top {len(selected_features)} features')

## 5. Cross Validation (Baseline)

In [ ]:
with mlflow.start_run(run_name='RandomForest_CV_Baseline'):

    X_raw = train_fe[selected_features].copy()
    y = train_fe[TARGET]
    cat_in_sel = [c for c in selected_features if train_fe[c].dtype == 'object']
    num_in_sel = [c for c in selected_features if train_fe[c].dtype != 'object']

    preprocessor = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), num_in_sel),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
            ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), cat_in_sel)
    ])

    baseline_params = {
        'n_estimators': 200,
        'max_depth': 15,
        'min_samples_split': 10,
        'min_samples_leaf': 5,
        'max_features': 'sqrt',
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }

    pipe = Pipeline([('pre', preprocessor), ('model', RandomForestClassifier(**baseline_params))])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipe, X_raw, y, cv=skf, scoring='roc_auc', n_jobs=-1)

    print(f'RF CV AUC: {cv_scores.mean():.5f} (+/- {cv_scores.std():.5f})')
    mlflow.log_params(baseline_params)
    mlflow.log_metric('cv_auc_mean', cv_scores.mean())
    mlflow.log_metric('cv_auc_std', cv_scores.std())

## 6. Hyperparameter Tuning (Optuna)

In [ ]:
with mlflow.start_run(run_name='RandomForest_Tuning'):

    X_raw = train_fe[selected_features].copy()
    y = train_fe[TARGET]
    cat_in_sel = [c for c in selected_features if train_fe[c].dtype == 'object']
    num_in_sel = [c for c in selected_features if train_fe[c].dtype != 'object']

    preprocessor = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), num_in_sel),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
            ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), cat_in_sel)
    ])

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 5, 30),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
            'class_weight': 'balanced',
            'random_state': 42, 'n_jobs': -1
        }
        pipe = Pipeline([('pre', preprocessor), ('model', RandomForestClassifier(**params))])
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        return cross_val_score(pipe, X_raw, y, cv=skf, scoring='roc_auc').mean()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=20, show_progress_bar=True)

    best_rf_params = study.best_params
    best_rf_params.update({'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1})
    print(f'Best RF AUC: {study.best_value:.5f}')
    mlflow.log_params(best_rf_params)
    mlflow.log_metric('best_cv_auc', study.best_value)

## 7. Overfit / Underfit Analysis

In [ ]:
X_raw = train_fe[selected_features].copy()
y = train_fe[TARGET]
cat_in_sel = [c for c in selected_features if train_fe[c].dtype == 'object']
num_in_sel = [c for c in selected_features if train_fe[c].dtype != 'object']

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_in_sel),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), cat_in_sel)
])

with mlflow.start_run(run_name='RandomForest_Overfit_Experiment'):
    overfit_pipe = Pipeline([('pre', preprocessor),
                              ('model', RandomForestClassifier(n_estimators=200, max_depth=None,
                                                                min_samples_leaf=1, random_state=42, n_jobs=-1))])
    train_aucs, val_aucs = [], []
    for tr_idx, val_idx in StratifiedKFold(3, shuffle=True, random_state=42).split(X_raw, y):
        overfit_pipe.fit(X_raw.iloc[tr_idx], y.iloc[tr_idx])
        train_aucs.append(roc_auc_score(y.iloc[tr_idx], overfit_pipe.predict_proba(X_raw.iloc[tr_idx])[:,1]))
        val_aucs.append(roc_auc_score(y.iloc[val_idx], overfit_pipe.predict_proba(X_raw.iloc[val_idx])[:,1]))
    print(f'[OVERFIT] Train: {np.mean(train_aucs):.5f} | Val: {np.mean(val_aucs):.5f}')
    mlflow.log_metric('train_auc', np.mean(train_aucs))
    mlflow.log_metric('val_auc', np.mean(val_aucs))
    mlflow.log_metric('overfit_gap', np.mean(train_aucs) - np.mean(val_aucs))
    mlflow.log_param('model_type', 'overfit_intentional')

with mlflow.start_run(run_name='RandomForest_Underfit_Experiment'):
    underfit_pipe = Pipeline([('pre', preprocessor),
                               ('model', RandomForestClassifier(n_estimators=5, max_depth=2, random_state=42))])
    cv_u = cross_val_score(underfit_pipe, X_raw, y, cv=StratifiedKFold(3, shuffle=True, random_state=42), scoring='roc_auc')
    print(f'[UNDERFIT] CV AUC: {cv_u.mean():.5f}')
    mlflow.log_metric('cv_auc_mean', cv_u.mean())
    mlflow.log_param('model_type', 'underfit_intentional')

## 8. Final Pipeline Training & Registration

In [ ]:
with mlflow.start_run(run_name='RandomForest_Final') as run:

    X_raw = train_fe[selected_features].copy()
    y_final = train_fe[TARGET]
    cat_in_sel = [c for c in selected_features if train_fe[c].dtype == 'object']
    num_in_sel = [c for c in selected_features if train_fe[c].dtype != 'object']

    preprocessor = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), num_in_sel),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
            ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), cat_in_sel)
    ])

    final_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(**best_rf_params))
    ])

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    final_cv = cross_val_score(final_pipeline, X_raw, y_final, cv=skf, scoring='roc_auc', n_jobs=-1)
    print(f'Final RF Pipeline CV AUC: {final_cv.mean():.5f} (+/- {final_cv.std():.5f})')

    final_pipeline.fit(X_raw, y_final)

    mlflow.log_metric('final_cv_auc_mean', final_cv.mean())
    mlflow.log_metric('final_cv_auc_std', final_cv.std())
    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path='rf_pipeline',
        registered_model_name='RandomForest_Fraud_Pipeline'
    )
    print(f'RF pipeline registered! Run ID: {run.info.run_id}')

with open('rf_selected_features.json', 'w') as f:
    json.dump(selected_features, f)